Preproses and load old model

In [1]:
from torchvision.io import decode_image
from torchvision import datasets, transforms
from torchvision.models import mnasnet0_5, MNASNet0_5_Weights
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

device = torch.device("cuda")



# Step 1: Initialize model with the best available weights
weights = MNASNet0_5_Weights.DEFAULT
model = mnasnet0_5(weights=weights)

model.classifier = nn.Sequential(
    nn.Flatten(),
    nn.Dropout(0.5),
    nn.Linear(1280, 128),
    nn.ReLU(),
    nn.Dropout(0.25),
    nn.Linear(128, 8),  # 8 Classes in DOES
    nn.Softmax(dim=1)
)


# Step 2: Initialize the inference transforms
preprocess = transforms.Compose([
    transforms.RandomHorizontalFlip(.5),
    transforms.RandomRotation(degrees=[1,30]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # Some common mean RGB values
                         std=[0.229, 0.224, 0.225])  # Common stddevs for RGB values

])
# Step 3: Apply inference preprocessing transforms
train_dataset = datasets.ImageFolder("DOES", transform=preprocess)
test_dataset = datasets.ImageFolder("TEST", transform=preprocess)
train_loader = DataLoader(train_dataset, batch_size=425, shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=425, shuffle=True,pin_memory=True)






Test Function

In [2]:
def get_accuracy_and_loss(model, loader, criterion):
  model.eval()
  my_loss = 0
  with torch.no_grad():
    correct = 0
    for data, target in loader:
      data, target = data.to(device), target.to(device)
      output = model(data)
      pred = output.argmax(dim=1)
      correct += pred.eq(target).sum().item()
      my_loss += criterion(output, target).item()
  return correct/len(loader.dataset), my_loss/len(loader.dataset)

Run test

In [3]:
import torch.nn.functional as F
import torch.optim as optim
best_val_loss = 1000000
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model.to(device)

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
time = 0

for epoch in range(1000):
    model.train()
    train_loss = 0
    correct = 0
    total_count = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)

        train_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total_count += data.size(0)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} done.")
    #Train Accuracy
    train_accuracy = correct / total_count
    train_accuracies.append(train_accuracy)
    print(f"Train accuracy: {train_accuracy}")

    #Train Loss
    train_loss = train_loss / total_count
    train_losses.append(train_loss)
    print(f"Train loss: {train_loss}")

    #Validation Accuracy and Loss
    val_accuracy, val_loss = get_accuracy_and_loss(model, test_loader, criterion)

    #Accuracy
    print(f"Val accuracy: {val_accuracy}")
    val_accuracies.append(val_accuracy)

    #Loss
    print(f"Val loss: {val_loss}")
    val_losses.append(val_loss)

    if val_loss <= best_val_loss:
        best_val_loss = val_loss
        torch.save({'model_state_dict': model.state_dict(),
            "val_loss": val_loss,
            "train_losses":train_losses,
            "val_losses":val_losses},"best_model_no_resize.pth")
        time = 0
    else:
        if time > 5:
            break
        else:
            time +=1

Epoch 0 done.
Train accuracy: 0.7954230235783634
Train loss: 0.0035036728117829544
Val accuracy: 0.4484169977127724
Val loss: 0.004384603169731562
Epoch 1 done.
Train accuracy: 0.9466658150229944
Train loss: 0.0031265436121473973
Val accuracy: 0.5107740459853136
Val loss: 0.004233923287860636
Epoch 2 done.
Train accuracy: 0.9626706328929119
Train loss: 0.0030895509763967993
Val accuracy: 0.48320693391115926
Val loss: 0.00429466569468162
Epoch 3 done.
Train accuracy: 0.9700525585809183
Train loss: 0.0030705712785145793
Val accuracy: 0.5058384495004213
Val loss: 0.004232116105454026
Epoch 4 done.
Train accuracy: 0.9761478940068619
Train loss: 0.003056444595900667
Val accuracy: 0.54544360178163
Val loss: 0.004146437417073157
Epoch 5 done.
Train accuracy: 0.9799620410248924
Train loss: 0.003047574546721932
Val accuracy: 0.5354520284097749
Val loss: 0.0041754076422602605
Epoch 6 done.
Train accuracy: 0.9829367107088108
Train loss: 0.0030400659115316746
Val accuracy: 0.5797520163717347
Val l

KeyboardInterrupt: 